# VishGuard — Colab demo (T5.2)

Launches the Streamlit UI on Colab T4 and exposes it publicly via an
ngrok tunnel. Use this notebook to record the Phase 5 demo video.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Add a Colab secret named `NGROK_AUTH_TOKEN` with your free ngrok
   token from [dashboard.ngrok.com](https://dashboard.ngrok.com/authtokens).
3. Set `VISHGUARD_REPO_URL` below (or set it as a Colab secret).

**Demo script (1–2 min):**
- 0:00 — narrate the vishing problem
- 0:15 — show Streamlit UI, upload a WAV
- 0:30 — transcript + spoof score + tactics populate
- 0:50 — risk score card + spoken briefing plays
- 1:10 — mention one known failure (see docs/FAILURES.md)
- 1:25 — show GitHub URL + close

In [ ]:
# Cell 1 — repo setup (self-contained; identical pattern to other notebooks)
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = os.environ.get('VISHGUARD_REPO_URL', 'https://github.com/PLACEHOLDER/vishguard.git')
DRIVE_SRC = '/content/drive/MyDrive/vishguard'
REPO_DIR  = Path('/content/vishguard')
ON_COLAB  = 'google.colab' in sys.modules or Path('/content').exists()

def sh(cmd, check=True):
    print('$', cmd)
    subprocess.run(cmd, shell=True, check=check)

if ON_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print('Drive mount skipped:', e)

    if 'PLACEHOLDER' not in REPO_URL:
        if REPO_DIR.exists():
            sh(f'cd {REPO_DIR} && git pull --ff-only')
        else:
            sh(f'git clone {REPO_URL} {REPO_DIR}')
    elif Path(DRIVE_SRC).exists():
        REPO_DIR.mkdir(parents=True, exist_ok=True)
        sh(f'rsync -a --delete --exclude .venv --exclude __pycache__ {DRIVE_SRC}/ {REPO_DIR}/')
    else:
        raise RuntimeError('Set VISHGUARD_REPO_URL or copy the repo to MyDrive/vishguard/')

    sh(f'pip install -q -e {REPO_DIR}')
    sh(f'pip install -q -r {REPO_DIR}/requirements.txt')
    try:
        import torch
        if torch.cuda.is_available():
            sh(f'pip install -q -r {REPO_DIR}/requirements-gpu.txt')
    except Exception as e:
        print('GPU extras skipped:', e)

    src = str(REPO_DIR / 'src')
    if src not in sys.path:
        sys.path.insert(0, src)

import vishguard
print('vishguard', vishguard.__version__)

## Cell 1b — Generate demo audio (optional)

Run this cell to synthesise a realistic vishing call WAV using SpeechT5
(the same model as the pipeline TTS stage). Because the voice is
machine-generated, the anti-spoof detector will flag it as synthetic —
giving the demo its most compelling result.

Skip this cell if you already have a WAV file to upload.

In [ ]:
# Cell 1b — generate a synthetic vishing call WAV using SpeechT5
# Speaker embedding approach mirrors notebooks/01_spike_antiSpoof.ipynb cell-4.
import pathlib, zipfile, numpy as np, torch, soundfile as sf
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
from huggingface_hub import snapshot_download

# Vishing script — covers authority, urgency, fear_intimidation,
# financial_manipulation, credential_harvesting, pretexting.
# Each segment kept under ~500 chars (SpeechT5 token limit).
SEGMENTS = [
    ('Hello. This is officer Daniel Reed from the Social Security '
     'Administration fraud division. We have detected suspicious activity '
     'linked to your social security number, which has now been suspended.'),
    ('You are facing a penalty of twenty four hundred dollars due to '
     'fraudulent use of your account. To avoid immediate arrest and a '
     'federal warrant being issued in your name, you must act now.'),
    ('Call our resolution hotline and provide your social security number '
     'and bank account details to verify your identity. '
     'You have twenty four hours before legal proceedings begin. '
     'This is your final warning.'),
]

DEMO_WAV    = pathlib.Path('/tmp/demo_call.wav')
SAMPLE_RATE = 16_000
device      = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

print('Loading SpeechT5 models...')
processor = SpeechT5Processor.from_pretrained('microsoft/speecht5_tts')
model     = SpeechT5ForTextToSpeech.from_pretrained('microsoft/speecht5_tts').to(device)
vocoder   = SpeechT5HifiGan.from_pretrained('microsoft/speecht5_hifigan').to(device)

print('Loading speaker embedding...')
xvec_repo = snapshot_download(
    repo_id='Matthijs/cmu-arctic-xvectors',
    repo_type='dataset',
    ignore_patterns=['*.py'],
)
zip_path    = pathlib.Path(xvec_repo) / 'spkrec-xvect.zip'
extract_dir = pathlib.Path('/tmp/cmu_arctic_xvect')
extract_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(extract_dir)

npy_files = sorted(extract_dir.rglob('*.npy'))
pt_files  = sorted(extract_dir.rglob('*.pt'))

if npy_files:
    xvec    = np.load(npy_files[7306 % len(npy_files)])
    speaker = torch.tensor(xvec).unsqueeze(0).to(device)
elif pt_files:
    data = torch.load(pt_files[0], map_location='cpu')
    if isinstance(data, torch.Tensor):
        speaker = data[min(7306, data.shape[0] - 1)].unsqueeze(0).to(device)
    elif isinstance(data, dict):
        speaker = torch.stack(list(data.values()))[0].unsqueeze(0).to(device)
    else:
        raise ValueError(f'unexpected pt type: {type(data)}')
else:
    print('WARNING: using normalized random embedding')
    speaker = torch.nn.functional.normalize(torch.randn(1, 512), dim=-1).to(device)

print(f'speaker shape: {speaker.shape}')

print('Synthesising segments...')
silence = np.zeros(int(SAMPLE_RATE * 0.35), dtype=np.float32)
parts   = []
for i, text in enumerate(SEGMENTS, 1):
    print(f'  {i}/{len(SEGMENTS)}: {text[:60]}...')
    inputs = processor(text=text, return_tensors='pt').to(device)
    with torch.no_grad():
        speech = model.generate_speech(
            inputs['input_ids'], speaker, vocoder=vocoder
        )
    parts.append(speech.cpu().numpy())
    parts.append(silence)

combined = np.concatenate(parts)
sf.write(str(DEMO_WAV), combined, samplerate=SAMPLE_RATE)
print(f'\nSaved: {DEMO_WAV}  ({len(combined)/SAMPLE_RATE:.1f}s)')

# Download the WAV to your local machine so you can re-upload it via
# the Streamlit file uploader during the demo recording.
from google.colab import files
files.download(str(DEMO_WAV))

In [ ]:
# Cell 2 — install pyngrok and configure auth token
import subprocess
subprocess.run(['pip', 'install', '-q', 'pyngrok'], check=True)

import os
from pyngrok import ngrok, conf

# Pull token from Colab secrets (preferred) or env var
try:
    from google.colab import userdata
    ngrok_token = userdata.get('NGROK_AUTH_TOKEN')
except Exception:
    ngrok_token = os.environ.get('NGROK_AUTH_TOKEN', '')

if not ngrok_token:
    raise RuntimeError(
        'Set NGROK_AUTH_TOKEN in Colab secrets (Secrets panel on the left) '
        'or as an environment variable. Free account at dashboard.ngrok.com.'
    )

conf.get_default().auth_token = ngrok_token
print('ngrok auth token configured')

In [ ]:
# Cell 3 — launch Streamlit in background, open ngrok tunnel
import subprocess, time
from pathlib import Path
from pyngrok import ngrok

REPO_DIR = Path('/content/vishguard')
PORT = 8501

_streamlit_proc = subprocess.Popen(
    [
        'streamlit', 'run',
        str(REPO_DIR / 'app.py'),
        f'--server.port={PORT}',
        '--server.headless=true',
        '--server.enableCORS=false',
        '--server.enableXsrfProtection=false',
    ],
    cwd=str(REPO_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

time.sleep(5)

tunnel = ngrok.connect(PORT, 'http')
public_url = tunnel.public_url

print('=' * 60)
print('VishGuard Streamlit UI:')
print(public_url)
print('=' * 60)
print('Open the URL above in a new tab to use the app.')
print('Run Cell 5 to shut down when done.')

In [ ]:
# Cell 4 (OPTIONAL) — CLI smoke test without the UI
# Uses the demo WAV from Cell 1b, or falls back to a LibriSpeech clip.
import subprocess, json
from pathlib import Path

REPO_DIR = Path('/content/vishguard')
AUDIO    = Path('/tmp/demo_call.wav')
OUT_DIR  = Path('/tmp/vishguard_out')
OUT_DIR.mkdir(exist_ok=True)

if not AUDIO.exists():
    from datasets import load_dataset
    import soundfile as sf
    ds   = load_dataset('librispeech_asr', 'clean', split='validation', streaming=True)
    item = next(iter(ds))
    sf.write(str(AUDIO), item['audio']['array'], item['audio']['sampling_rate'])
    print(f'Saved fallback sample: {AUDIO}')

result = subprocess.run(
    ['python', '-m', 'vishguard.cli', '--audio', str(AUDIO), '--out-dir', str(OUT_DIR)],
    cwd=str(REPO_DIR),
    capture_output=True,
    text=True,
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

report_files = list(OUT_DIR.glob('*.json'))
if report_files:
    print(json.dumps(json.loads(report_files[-1].read_text()), indent=2))

In [ ]:
# Cell 5 — shutdown (run after recording to free port + kill tunnel)
from pyngrok import ngrok

ngrok.kill()

try:
    _streamlit_proc.terminate()
    _streamlit_proc.wait(timeout=5)
    print('Streamlit stopped')
except Exception as e:
    print('Shutdown:', e)